# Validate the locked larger-development protocol

This notebook independently reconstructs and validates the locked 256-source development selection in a fresh Colab checkout. It verifies all bound hashes, the shuffled-image mapping, and the test-set seal. It performs **no model inference** and does not require a GPU.

Commit and push the protocol implementation first, then paste its full 40-character commit below.


In [ ]:
# 1. Set the full pushed commit containing the locked protocol.
REPOSITORY_COMMIT = "REPLACE_WITH_FULL_40_CHARACTER_COMMIT"
REPOSITORY_URL = "https://github.com/dizza01/VLM.git"
REPOSITORY_ROOT = "/content/VLM-larger-development-lock"
PROJECT_ROOT = f"{REPOSITORY_ROOT}/gi_vqa_research"

if len(REPOSITORY_COMMIT) != 40 or REPOSITORY_COMMIT.startswith("REPLACE"):
    raise RuntimeError("Paste the full pushed Git commit before continuing")


In [ ]:
# 2. Create a fresh exact checkout.
from pathlib import Path
import shutil
import subprocess

checkout = Path(REPOSITORY_ROOT)
if checkout.exists():
    shutil.rmtree(checkout)
subprocess.run(["git", "clone", REPOSITORY_URL, REPOSITORY_ROOT], check=True)
subprocess.run(["git", "checkout", REPOSITORY_COMMIT], cwd=checkout, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=checkout, check=True,
    capture_output=True, text=True,
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=checkout, check=True,
    capture_output=True, text=True,
).stdout
print("Commit:", observed)
print("Checkout clean:", not bool(status))
if observed != REPOSITORY_COMMIT or status:
    raise RuntimeError("Checkout verification failed")
%cd /content/VLM-larger-development-lock/gi_vqa_research


In [ ]:
# 3. Install the project data dependencies and compatible fsspec.
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-e", ".[data]", "fsspec==2024.12.0",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
import datasets
import fsspec
print("datasets:", datasets.__version__)
print("fsspec:", fsspec.__version__)
if datasets.__version__ != "3.3.2" or fsspec.__version__ != "2024.12.0":
    raise RuntimeError("Pinned data dependencies were not installed")
print("Installation completed")


In [ ]:
# 4. Materialize the pinned grouped-split artifacts.
import os
import subprocess
import sys

environment = os.environ.copy()
environment["PYTHONPATH"] = "src"
completed = subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.cli", "materialize-splits",
        "--manifest", "protocols/study1/grouped_split_manifest.json",
        "--project-root", ".",
    ],
    cwd=PROJECT_ROOT,
    env=environment,
    check=False,
    capture_output=True,
    text=True,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError("Grouped-split materialization failed")


In [ ]:
# 5. Reconstruct the selection into a temporary path and compare it exactly.
import json
import subprocess
import sys
from pathlib import Path

temporary_selection = Path(PROJECT_ROOT) / "runs/lock-validation/selection.json"
temporary_selection.parent.mkdir(parents=True, exist_ok=True)
temporary_selection.unlink(missing_ok=True)
completed = subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.larger_development", "select",
        "--project-root", ".",
        "--output", str(temporary_selection.relative_to(PROJECT_ROOT)),
    ],
    cwd=PROJECT_ROOT,
    env=environment,
    check=False,
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError("Selection reconstruction failed")
tracked = json.loads(
    (Path(PROJECT_ROOT) / "protocols/study1/larger_development_selection.json")
    .read_text(encoding="utf-8")
)
reconstructed = json.loads(temporary_selection.read_text(encoding="utf-8"))
if reconstructed != tracked:
    raise RuntimeError("Reconstructed selection differs from the tracked lock")
print("Exact selection reconstruction PASS")


In [ ]:
# 6. Validate the complete machine-readable protocol and test-set seal.
completed = subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.larger_development", "check",
        "--project-root", ".",
        "--protocol", "protocols/study1/larger_development_protocol.json",
    ],
    cwd=PROJECT_ROOT,
    env=environment,
    check=True,
    capture_output=True,
    text=True,
)
result = json.loads(completed.stdout)
print(json.dumps(result, indent=2, sort_keys=True))
if result["status"] != "PASS":
    raise RuntimeError("Protocol validation failed")
if result["development_items"] != 256:
    raise RuntimeError("Unexpected development item count")
if result["test_partition_accessed"] is not False:
    raise RuntimeError("Test partition seal failed")
print("\nLarger-development lock validation PASS")


A PASS here validates only the preregistration, deterministic selection and test-set seal. Do not start the larger inference experiment until its restart-safe runner and evidence packager have been implemented and tested.
